<a href="https://colab.research.google.com/github/wesoho/Bringing-Old-Photos-Back-to-Life/blob/master/MiniMax_H3_ComfyUI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax-H3 · Colab 一键部署（ComfyUI + 公网 API + Web 界面）

在 **Colab 免费 T4** 上一键跑起 MiniMax-H3（视频 + 同步音频），启动后同时获得：

- **Web 工作台**：ComfyUI 界面（cloudflared 公网地址直接打开），可视化搭/跑工作流
- **HTTP API**：ComfyUI 原生接口 `POST /prompt`、`GET /history/{id}`、`GET /view`，你的程序可直接调用
- **ComfyUI-Manager**：界面内管理节点/模板

## 免费档位实测参考（来自社区 T4 实测 gist，本 notebook 继承其全部选型）

| 配置 | 分辨率 | T4 出片耗时 |
|---|---|---|
| 6 步（推荐） | 864x480 | ~35 分钟 |
| 4 步 | 960x544 | ~40 分钟 |

**关键事实（都是踩坑换来的）**：
1. Colab 免费档的瓶颈是 **主机内存 12.7GB**，不是 T4 的 14.9GB 显存；
2. H3 的 DiT 是 bf16 训练，**必须 `--bf16-unet`**（fp16 会数学溢出出 NaN，T4 模拟 bf16 较慢）；
3. 采样用 **GGUF 量化版必须声明 arch=`wan`**（Abiray 版可用，unsloth 版会被 ComfyUI-GGUF 拒载）；
4. 帧数必须满足 `n % 17 == 5`（24fps 下 5.2 秒 = 124 帧），宽高须被 32 整除；
5. 4-8 步靠 **Turbo LoRA（strength=1.0）** 实现，必须配作者节点包运行时注入；
6. 权重合计约 32GB（DiT 10.8 + 文本编码器 14.6 + 双 VAE 5.5 + LoRA 0.8），磁盘够但每次会话要重下。

## 使用顺序

1. Runtime → **T4 GPU**，然后全部按序运行；
2. Stage 0 勾选许可（license-gated 模型，先读 [官方 LICENSE](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE)）；
3. Stage 3 跑完会打印 **公网 URL**：浏览器开它就是 Web 界面，程序请求它就是 API；
4. 在 Web 界面跑通一次工作流 → 导出 (API) 格式 → 粘到 Stage 4 客户端模板即可程序化调用。

## Credits / 选型出处

- [ComfyUI](https://github.com/Comfy-Org/ComfyUI) · [ComfyUI-Manager 官方 Colab notebook](https://github.com/ltdrdata/ComfyUI-Manager)（cloudflared 启动模式）
- [ComfyUI-GGUF](https://github.com/city96/ComfyUI-GGUF) · [Larryvrh/ComfyUI-MiniMax-H3-Turbo](https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo)
- 权重：[Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3) · [Abiray/MiniMax-H3-Pruned-GGUF](https://huggingface.co/Abiray/MiniMax-H3-Pruned-GGUF) · [larryvrh/MiniMax-H3-Turbo-Lora](https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora)
- T4 可行性与全部工程选型来自社区实测 gist [hirokazu/minimax_h3_on_colab_t4](https://gist.github.com/hirokazu/f94c539ba208784c9ec9cfc18acffd73)，commit 均沿用其验证过的组合



In [ ]:
# @title Stage 0 · 配置与许可确认（Colab 表单可直接改）
import os, sys, shutil, time
from pathlib import Path

# ---------- MiniMax-H3 许可（license-gated，勾选后才继续下载） ----------
# 请先阅读 https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE
I_HAVE_READ_AND_ACCEPTED_MINIMAX_H3_LICENSE = True  # @param {type:"boolean"}
MINIMAX_LICENSE_BASIS = "applicable_territory"  # @param ["applicable_territory", "separate_minimax_authorization"]
I_CONFIRM_RUNTIME_AND_USE_LOCATIONS_ARE_COVERED = True  # @param {type:"boolean"}
I_CONFIRM_ACCEPTABLE_USE_AND_OUTPUT_RULES = True  # @param {type:"boolean"}
I_ACCEPT_RESPONSIBILITY_FOR_OUTPUTS = True  # @param {type:"boolean"}
RUN_MODEL_DOWNLOAD = True  # @param {type:"boolean"}

HF_TOKEN = ""  # @param {type:"string"}   # 权重仓库若需登录，粘贴你的 HF token；公开仓库留空即可

# ---------- 生成参数（沿用 T4 实测推荐值） ----------
DIT_CHOICE = "abiray_Q4_K_M"  # @param ["abiray_Q4_K_M", "abiray_Q3_K_M"]  # Q3 更省显存、质量略降
GEAR = "0.4"  # @param ["0.2", "0.3", "0.4", "0.5", "1.0"]             # 分辨率档位（见下表）
CLIP_SECONDS = 5.2  # @param {type:"number"}
STEPS = 6  # @param {type:"integer"}                                        # Turbo LoRA 有效区间 4-8，6-8 更好
FPS = 24
MOUNT_DRIVE = True  # @param {type:"boolean"}                               # 挂 Drive：输出视频持久化
ENABLE_COMFYUI_MANAGER = True  # @param {type:"boolean"}

# ---------- 校验 ----------
license_ok = all((I_HAVE_READ_AND_ACCEPTED_MINIMAX_H3_LICENSE,
                  MINIMAX_LICENSE_BASIS in ("applicable_territory", "separate_minimax_authorization"),
                  I_CONFIRM_RUNTIME_AND_USE_LOCATIONS_ARE_COVERED,
                  I_CONFIRM_ACCEPTABLE_USE_AND_OUTPUT_RULES,
                  I_ACCEPT_RESPONSIBILITY_FOR_OUTPUTS))
if not license_ok:
    raise RuntimeError("请先阅读 MiniMax-H3 LICENSE 并勾选全部 4 项确认")

GEAR_TABLE = {"0.2": (608, 352), "0.3": (736, 416), "0.4": (864, 480),
              "0.5": (960, 544), "1.0": (1344, 768)}
W, H = GEAR_TABLE[GEAR]

def frame_count(seconds):
    """H3 帧数必须满足 n % 17 == 5，否则模型会静默取整成别的几何"""
    n = max(5, round(seconds * FPS))
    return n + (5 - (n % 17)) % 17

FRAMES = frame_count(CLIP_SECONDS)
assert W % 32 == 0 and H % 32 == 0, "宽高须被 32 整除"
assert (FRAMES - 5) % 17 == 0, "帧数不合法"

# ---------- 环境 ----------
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无GPU!请切换 T4 运行时"
ram = shutil.disk_usage("/content")
print(f"GPU: {gpu} | torch {torch.__version__} | cuda {torch.cuda.is_available()}")
print(f"目标几何: {W}x{H} / {FRAMES} 帧 / {STEPS} 步 / {CLIP_SECONDS}s @{FPS}fps")
print(f"磁盘剩余: {ram.free/1e9:.0f} GB")

if MOUNT_DRIVE:
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    print("Drive 已挂载")

PUB = None  # Stage 3 会写入公网地址
print("Stage 0 完成")


## Stage 1 · 安装 ComfyUI 与节点（commit 全部固定）

全部沿用 T4 实测 gist 验证过的组合：ComfyUI / ComfyUI-GGUF / Turbo 节点各自 pin 到指定 commit；ComfyUI-GGUF 需要一行补丁把 `minimax_h3` 加进架构白名单（否则 GGUF 会在读张量之前就被拒）。Colab 自带 torch 不动它。



In [ ]:
# @title Stage 1 · 安装 ComfyUI + GGUF/Turbo/Manager 节点（约 2-3 分钟）
import subprocess, sys, re
from pathlib import Path

COMFY_DIR   = Path("/content/ComfyUI")
COMFY_COMMIT   = "9a9fdb10ed144ce760d9682cb247526ea23cc525"   # gist 实测组合
GGUF_COMMIT    = "6ea2651e7df66d7585f6ffee804b20e92fb38b8a"
TURBO_COMMIT   = "55fee864dd7b2976b1c4ce3c3d5f7968f181409f"

def sh(cmd, cwd=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

def clone_pin(repo, dest, commit):
    if not dest.exists():
        sh(["git", "clone", "--filter=blob:none", repo, str(dest)])
    sh(["git", "-C", str(dest), "checkout", commit])

clone_pin("https://github.com/Comfy-Org/ComfyUI.git", COMFY_DIR, COMFY_COMMIT)
CUSTOM = COMFY_DIR / "custom_nodes"
clone_pin("https://github.com/city96/ComfyUI-GGUF.git", CUSTOM/"ComfyUI-GGUF", GGUF_COMMIT)
clone_pin("https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git", CUSTOM/"ComfyUI-MiniMax-H3-Turbo", TURBO_COMMIT)

if ENABLE_COMFYUI_MANAGER:
    mgr = CUSTOM / "ComfyUI-Manager"
    if not mgr.exists():
        sh(["git", "clone", "--depth", "1", "https://github.com/ltdrdata/ComfyUI-Manager", str(mgr)])
    sh([sys.executable, "-m", "pip", "install", "-q", "GitPython"])

# ---- ComfyUI-GGUF 架构白名单补丁：加入 minimax_h3（幂等） ----
target = None
for f in (CUSTOM/"ComfyUI-GGUF").rglob("*.py"):
    t = f.read_text()
    if "IMG_ARCH_LIST" in t and "qwen_image" in t:
        target = f; break
if target is None:
    raise RuntimeError("未找到 IMG_ARCH_LIST 定义文件（节点版本变了？请把 GGUF_COMMIT 对齐最新）")
t = target.read_text()
if "minimax_h3" in t:
    print("架构补丁已存在，跳过")
else:
    patched, n = re.subn(r'("qwen_image")(\s*)}', r'\1, "minimax_h3"\2}', t, count=1)
    assert n == 1, f"anchor 替换失败（n={n}），请手动检查 {target}"
    target.write_text(patched)
    print("已打架构补丁:", target)

# ---- 依赖（Colab 自带 torch，版本满足则不会重装） ----
sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_DIR/"requirements.txt")])
sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(CUSTOM/"ComfyUI-GGUF"/"requirements.txt")])

import torch
print("torch 保持不变:", torch.__version__)
print("Stage 1 完成")


## Stage 2 · 下载权重（首次约 32GB，Colab 出口带宽快）

| 组件 | 仓库 | 大小 | 落位 |
|---|---|---|---|
| DiT（采样） | `Abiray/MiniMax-H3-Pruned-GGUF`（声明 arch=wan） | 10.8 GB | `models/unet/` |
| 文本编码器 | `Comfy-Org/MiniMax-H3` | 14.6 GB | `models/text_encoders/` |
| 视频 VAE | `Comfy-Org/MiniMax-H3` | 4.9 GB | `models/vae/` |
| 音频 VAE | `Comfy-Org/MiniMax-H3` | 0.6 GB | `models/vae/` |
| Turbo LoRA | `larryvrh/MiniMax-H3-Turbo-Lora` | 0.8 GB | `models/loras/` |

说明：unsloth 的 GGUF 不声明 `general.architecture`，ComfyUI-GGUF 会以 `Unknown model architecture!` 拒载，所以这里只提供 Abiray 两档。`hf_hub_download` 自带 LFS 完整性校验，不必手写 SHA。



In [ ]:
# @title Stage 2 · 下载权重 + 官方工作流模板（10-20 分钟）
from huggingface_hub import hf_hub_download, login, list_repo_files

if HF_TOKEN.strip():
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)

DIT = {
    "abiray_Q4_K_M": ("Abiray/MiniMax-H3-Pruned-GGUF", "MiniMax-H3-FL2VA-Pruned-Q4_K_M.gguf"),
    "abiray_Q3_K_M": ("Abiray/MiniMax-H3-Pruned-GGUF", "MiniMax-H3-FL2VA-Pruned-Q3_K_M.gguf"),
}[DIT_CHOICE]

MODELS = COMFY_DIR / "models"

def get(repo, fname, local_dir):
    print(f"↓ {repo} :: {fname}")
    hf_hub_download(repo_id=repo, filename=fname, local_dir=str(local_dir))

if RUN_MODEL_DOWNLOAD:
    get(DIT[0], DIT[1], MODELS/"unet")                       # -> models/unet/*.gguf
    get("larryvrh/MiniMax-H3-Turbo-Lora",
        "minimax_h3_turbo_v4_step600_ema.safetensors", MODELS/"loras")
    get("Comfy-Org/MiniMax-H3", "vae/minimax_h3_video_vae_fp16.safetensors", MODELS)
    get("Comfy-Org/MiniMax-H3", "vae/minimax_h3_audio_vae_fp32.safetensors", MODELS)
    get("Comfy-Org/MiniMax-H3",
        "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", MODELS)
else:
    print("跳过下载（复用已有权重）")

# 官方工作流模板自动发现并放入用户工作流目录（Web 界面里可直接打开）
try:
    wf_dir = COMFY_DIR / "user/default/workflows"
    wf_dir.mkdir(parents=True, exist_ok=True)
    wfs = [f for f in list_repo_files("Comfy-Org/MiniMax-H3") if f.endswith(".json")]
    for f in wfs:
        hf_hub_download("Comfy-Org/MiniMax-H3", f, local_dir=str(wf_dir))
    print("工作流模板已就位:", wfs or "(仓库无 json 模板)")
except Exception as e:
    print("模板拉取失败(不影响主流程):", e)

tot = sum(p.stat().st_size for p in MODELS.rglob("*") if p.is_file())
print(f"models 合计 {tot/1e9:.1f} GB")
print("Stage 2 完成")


## Stage 3 · 启动 ComfyUI + cloudflared 公网隧道

沿用 ComfyUI-Manager 官方 notebook 的模式：装 cloudflared .deb → 后台起 ComfyUI（**必须 `--bf16-unet`**）→ 等端口就绪 → 起隧道 → 打印 `https://xxx.trycloudflare.com`。

- 浏览器打开该地址 = **Web 工作台**（第一次加载界面稍慢，正常）
- 程序请求该地址 = **API**（`/prompt` `/history` `/view`）
- 输出视频自动转存到 Google Drive `MyDrive/H3-CUI/output/`（挂载了 Drive 时）

> 免费档注意：T4 上 bf16 是模拟执行（无原生支持），单步 200 秒级是正常现象；6 步 864x480 全程约 35 分钟。若遇 OOM，按顺序尝试：GEAR 降到 0.3 → DIT 换 `abiray_Q3_K_M` → 换 L4/A100 运行时。



In [ ]:
# @title Stage 3 · 启动服务 + 公网隧道（跑完看打印的 URL）
import subprocess, sys, socket, time, re, shutil, threading
from pathlib import Path

PORT = 8188
COMFY_LOG = Path("/content/comfyui.log")
TUNNEL_LOG = Path("/content/tunnel.log")

# 1) cloudflared（官方 notebook 同款安装方式）
if not shutil.which("cloudflared"):
    sh(["wget", "-q", "-P", str(Path.home()),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"])
    sh(["dpkg", "-i", str(Path.home()/"cloudflared-linux-amd64.deb")])

# 2) 输出目录转存 Drive（可选）
if Path("/content/drive/MyDrive").is_dir():
    d_out = Path("/content/drive/MyDrive/H3-CUI/output")
    d_out.mkdir(parents=True, exist_ok=True)
    real_out = COMFY_DIR/"output"
    if not real_out.exists():
        real_out.parent.mkdir(parents=True, exist_ok=True)
        real_out.symlink_to(d_out)
    print("输出目录 -> Drive:", d_out)

# 3) 后台启动 ComfyUI（H3 必须 bf16，fp16 会溢出 NaN）
log = open(COMFY_LOG, "w")
proc = subprocess.Popen(
    [sys.executable, "main.py", "--dont-print-server",
     "--listen", "127.0.0.1", "--port", str(PORT), "--bf16-unet"],
    cwd=str(COMFY_DIR), stdout=log, stderr=subprocess.STDOUT)

# 4) 等 8188 就绪（模型多时加载要几分钟）
print("等待 ComfyUI 就绪…")
ready = False
for _ in range(300):
    s = socket.socket(); r = s.connect_ex(("127.0.0.1", PORT)); s.close()
    if r == 0: ready = True; break
    if proc.poll() is not None:
        print("ComfyUI 进程退出！日志尾部:"); print(COMFY_LOG.read_text()[-3000:]); break
    time.sleep(2)
if not ready: raise SystemExit("ComfyUI 未在 10 分钟内就绪")
print("ComfyUI 就绪: 127.0.0.1:8188")

# 5) 起隧道并抓取公网地址
tl = open(TUNNEL_LOG, "w")
tp = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
                      stdout=tl, stderr=subprocess.STDOUT)
PUB = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", TUNNEL_LOG.read_text())
    if m: PUB = m.group(0); break

print("=" * 60)
print("Web 界面 :", PUB + "/" if PUB else "(隧道未就绪，查看 /content/tunnel.log)")
print("API 根地址:", PUB)
print("=" * 60)

# 6) 自检
import requests
print("本机自检:", requests.get(f"http://127.0.0.1:{PORT}/system_stats").json()["devices"][0]["name"], "在线")
if PUB:
    print("公网自检: HTTP", requests.get(PUB + "/system_stats", timeout=30).status_code)
print("日志文件:", COMFY_LOG, "(报错时 tail 它)")


## Stage 4 · 程序化调用（ComfyUI 原生 API）

ComfyUI 的 API 三件套（对公网地址发请求即可）：

| 动作 | 接口 | 说明 |
|---|---|---|
| 提交任务 | `POST /prompt` | body `{"prompt": <工作流API JSON>, "client_id": "任意"}`，返回 `prompt_id` |
| 查状态 | `GET /history/{prompt_id}` | `status.completed` 为真时结束，`outputs` 里带产物文件名 |
| 取文件 | `GET /view?filename=..&subfolder=..&type=output` | 下载 MP4 |

**工作流 API JSON 怎么来**：在 Web 界面把工作流跑通一次 → 菜单「Workflow/工作流 → Export (API)」导出 → 粘贴到下面 `WORKFLOW_API`。T4 上一单 30-40 分钟，客户端要设长超时。

> 公网 trycloudflare 地址无鉴权、每次重启会变。长期使用：换成 Cloudflare 命名隧道或 frp 连你自己的服务器；给业务调用方建议通过你自己的后端转发，不要把裸地址写死。



In [ ]:
# @title Stage 4 · API 客户端模板（含连通性自测）
import requests, time, uuid

BASE = PUB or "http://127.0.0.1:8188"     # 外部机器调用时替换为你的公网地址

# ① Web 界面「Export (API)」导出后，粘贴成 dict（或 json.loads('''…''')）
WORKFLOW_API = None   # @param  ← 粘贴后改这里

def queue(base, workflow, client_id=None):
    cid = client_id or uuid.uuid4().hex
    r = requests.post(f"{base}/prompt", json={"prompt": workflow, "client_id": cid}, timeout=60)
    r.raise_for_status()
    return r.json()["prompt_id"], cid

def wait_done(base, prompt_id, timeout=7200, poll=10):
    """T4 上 6 步约 35 分钟，默认给 2 小时"""
    t0 = time.time()
    while time.time() - t0 < timeout:
        h = requests.get(f"{base}/history/{prompt_id}", timeout=60).json()
        if prompt_id in h:
            st = h[prompt_id].get("status", {})
            if st.get("completed") or st.get("status_str") == "error":
                return h[prompt_id].get("outputs", {}), st
        time.sleep(poll)
    raise TimeoutError(f"{timeout}s 未完成")

def grab_urls(base, outputs):
    urls = []
    for node_out in outputs.values():
        for item in (node_out.get("gifs", []) + node_out.get("videos", [])
                     + node_out.get("images", [])):
            urls.append(f"{base}/view?filename={item['filename']}"
                        f"&subfolder={item.get('subfolder','')}&type={item.get('type','output')}")
    return urls

# ② 连通性自测（不提交任务）
print("API 连通:", requests.get(f"{BASE}/system_stats", timeout=30).status_code, BASE)

# ③ 正式调用（粘好 WORKFLOW_API 后重跑本 cell）
if WORKFLOW_API:
    pid, cid = queue(BASE, WORKFLOW_API)
    print("job:", pid)
    outs, st = wait_done(BASE, pid)
    print("status:", st)
    for u in grab_urls(BASE, outs):
        print("MP4:", u)
        print("  curl -o out.mp4", "'" + u + "'")
else:
    print("提示: 先在 Web 界面跑通工作流并 Export (API)，把 JSON 粘到 WORKFLOW_API 再重跑本 cell")


## Stage 5 · 保活与运维

- Colab 闲置约 90 分钟断线：运行下面 cell 注入每 60s 模拟点击的 JS，只防**闲置**，防不了免费档 4–6 小时的会话上限；
- 断线后一切重来（权重重下、URL 变）：把 Stage 3 重新跑一遍即可；客户端不要写死 URL；
- Colab 条款不允许把免费实例当长期托管服务，此 notebook 定位是**开发验证**；正式跑量建议 AutoDL/RunPod 或 MiniMax 官方 API；
- 排错顺序：`/content/comfyui.log` → `/content/tunnel.log` → 重跑 Stage 3。



In [ ]:
# @title Stage 5 · 防闲置保活（可选）
from IPython.display import Javascript
Javascript("""
function ConnectButton(){
  var b = document.querySelector('colab-connect-button');
  if(b && b.shadowRoot) b.shadowRoot.querySelector('#connect').click();
}
setInterval(ConnectButton, 60000);
""")
print("保活 JS 已注入（仅防闲置断开）")
print("提醒: URL 在, 服务在; cell 停/会话断, API 即失效。重连后重跑 Stage 3。")
